In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
conn = duckdb.connect()

account_level = conn.sql("""
WITH usage_summary AS (
    SELECT
        s.account_id,
        COUNT(DISTINCT fu.feature_name) AS features_adopted,
        SUM(fu.usage_count) AS total_usage_count,
        SUM(fu.error_count) * 1.0 /
            NULLIF(SUM(fu.usage_count), 0) AS avg_error_rate
    FROM read_csv_auto('../data/raw/ravenstack_feature_usage.csv') fu
    JOIN read_csv_auto('../data/raw/ravenstack_subscriptions.csv') s
        ON fu.subscription_id = s.subscription_id
    GROUP BY 1
),

support_summary AS (
    SELECT
        account_id,
        COUNT(*) AS ticket_count,
        AVG(resolution_time_hours) AS avg_resolution_time,
        AVG(CASE WHEN escalation_flag THEN 1 ELSE 0 END) AS escalation_rate,
        AVG(satisfaction_score) AS avg_satisfaction_score
    FROM read_csv_auto('../data/raw/ravenstack_support_tickets.csv')
    GROUP BY 1
)

SELECT
    a.account_id,
    CASE WHEN a.churn_flag THEN 1 ELSE 0 END AS churn_flag,
    a.industry,
    a.country,
    a.plan_tier,

    COALESCE(u.features_adopted, 0) AS features_adopted,
    COALESCE(u.total_usage_count, 0) AS total_usage_count,
    COALESCE(u.avg_error_rate, 0) AS avg_error_rate,

    COALESCE(s.ticket_count, 0) AS ticket_count,
    COALESCE(s.avg_resolution_time, 0) AS avg_resolution_time,
    COALESCE(s.escalation_rate, 0) AS escalation_rate,
    COALESCE(s.avg_satisfaction_score, 0) AS avg_satisfaction_score

FROM read_csv_auto('../data/raw/ravenstack_accounts.csv') a
LEFT JOIN usage_summary u
    ON a.account_id = u.account_id
LEFT JOIN support_summary s
    ON a.account_id = s.account_id
""").df()

account_level.head()

In [ ]:
numeric_corr = account_level[
    [
        "churn_flag",
        "features_adopted",
        "total_usage_count",
        "avg_error_rate",
        "ticket_count",
        "avg_resolution_time",
        "escalation_rate",
        "avg_satisfaction_score"
    ]
].corr()

numeric_corr

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    numeric_corr,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Correlation Matrix - Churn Analysis")
plt.tight_layout()

plt.show()

In [ ]:
numeric_corr["churn_flag"] \
    .sort_values(ascending=False)

In [ ]:
encoded = pd.get_dummies(
    account_level,
    columns=["industry", "country", "plan_tier"]
)

segment_corr = encoded.corr(numeric_only=True)

segment_corr["churn_flag"] \
    .sort_values(ascending=False) \
    .head(20)